# Databricks SQL Notebook - E-Commerce Analytics Pipeline
This notebook contains syntactically valid SQL queries with keyword casing violations and performance anti-patterns for LLM code review.

In [1]:
%py
# PySpark session initialization - ignored by SQL reviewer agent
import pyspark.sql.functions as F
print("Databricks environment initialized.")

In [ ]:
%sql
-- Cell 1: Customer Sales Summary (Valid AST, lowercase keywords, SELECT * anti-pattern)
select c.cust_id, c.cust_name, o.order_id, o.amount, o.order_date
from workspace.customers c
left join workspace.orders o on c.cust_id = o.cust_id
where c.status = 'ACTIVE' and date_format(o.order_date, 'yyyy') = '2026'
group by c.cust_id, c.cust_name, o.order_id, o.amount, o.order_date
having count(o.order_id) >= 1
order by o.amount desc;

In [3]:
%sql
-- Cell 2: Product Inventory Breakdown (Valid AST, mixed-case keywords)
sElEcT p.product_id, p.product_name, p.category, p.unit_price, i.stock_quantity
fRoM workspace.products p
lEfT jOiN workspace.inventory i oN p.product_id = i.product_id
wHeRe p.is_discontinued = false aNd i.stock_quantity < 50
oRdEr bY i.stock_quantity aSc;

In [ ]:
%sql
-- Cell 4: Super Complex query testing compound keywords (partition by, rows between, left outer join, union all, etc.)
with complex_cte as (
    select 
        cust_id,
        first_value(txn_amt) over (partition by cust_id order by txn_dt rows between unbounded preceding and unbounded following) as first_amt,
        sum(txn_amt) as total_amt
    from workspace.txns
    where status is not null and category not in ('VOID', 'REFUND')
    group by cust_id, txn_dt, txn_amt
)
select 
    c.cust_id as cust, 
    c.total_amt as amt, 
    u.name
from complex_cte c
left outer join workspace.users u on c.cust_id = u.id
union all
select 
    c.cust_id as cust, 
    c.total_amt as amt, 
    'UNKNOWN' as name
from complex_cte c
where c.cust_id not in (select id from workspace.users)
order by amt desc;

In [ ]:
use catalog my_catalog;
use schema my_schema;

In [ ]:
%sql

MERGE INTO workspace.customer_summary AS target
USING workspace.orders AS source
ON target.customer_id = source.customer_id

WHEN MATCHED THEN
    UPDATE SET
        target.total_revenue = source.total_revenue,
        target.last_order_date = source.last_order_date

WHEN NOT MATCHED THEN
    INSERT (
        customer_id,
        total_revenue,
        last_order_date
    )
    VALUES (
        source.customer_id,
        source.total_revenue,
        source.order_date
    );

In [ ]:
%python
# Complex PySpark cell containing sql queries
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Dynamic query using variable assignment with casing errors
join_type = "left"
sql_str = f"select u.id as user_id, u.name, o.amt from users u {join_type} join orders o on u.id = o.user_id"
df1 = spark.sql(sql_str)

# CTE and Window Function query with lowercase keyword violations
spark.sql("""
with ranked_payments as (
    select 
        pay_id,
        txn_amt,
        row_number() over (partition by user_id order by pay_date desc) as rank
    from workspace.payments
)
select * from ranked_payments where rank = 1
""")
